# Derivative of the matter power spectrum

The smallest use of DISCO-EB's derivatives: $\partial P(k) / \partial \Omega_m$ today by forward-mode automatic differentiation, checked against a finite difference.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from discoeb import LCDM, conformal_ages, matter_power_spectrum, solve_eb

K = np.geomspace(1.0e-4, 1.0, 64)  # wave numbers, 1/Mpc

## The cosmology

The Planck 2018 best fit, flat, with three massless neutrino flavours.

In [2]:
h = 0.6732117
planck = LCDM(
    Omegam=(0.02238280 + 0.1201075) / h**2,
    Omegab=0.02238280 / h**2,
    H0=100.0 * h,
    A_s=2.100549e-9,
    n_s=0.9660499,
    Tcmb=2.7255,
    YHe=0.2454006,
    Neff=3.046,
)

## From $\Omega_m$ to $P(k)$

`power` maps $\Omega_m$ to $P(k)$ today. `conformal_ages` gives the conformal age $\tau_0$, `solve_eb` keeps the four quantities `matter_power_spectrum` needs at that time, and `matter_power_spectrum` turns them into $P(k)$.

$\Omega_b$, $H_0$ and flatness are held, so more matter means less dark energy.

In [3]:
def power(Omegam):
    """``P(k)`` today in Mpc^3, ``(len(K),)``, of the Planck cosmology at this Omega_m."""

    cosmology = planck.replace(Omegam=Omegam)
    tau0 = conformal_ages(cosmology)[0]
    values = solve_eb(  # (cosmology, name, k, tau)
        cosmology, K, tau0.reshape(1), keep=("clxc", "clxb", "z", "vb")
    )
    clxc, clxb, z, vb = jnp.unstack(values[0, :, :, -1])
    return matter_power_spectrum(
        K,
        clxc=clxc,
        clxb=clxb,
        z=z,
        vb=vb,
        tau0=tau0,
        Omegam=cosmology.Omegam,
        Omegab=cosmology.Omegab,
        A_s=cosmology.A_s,
        n_s=cosmology.n_s,
        k_p=cosmology.k_p,
    )

## The derivative

`jax.jacfwd` of `power` gives $\partial P / \partial \Omega_m$ at every wave number from one forward-sensitivity launch. Pass `tau0` from `conformal_ages` to `matter_power_spectrum` as above: it adds the term from the shift of today's conformal time with the parameter.

In [4]:
Omegam = jnp.asarray(planck.Omegam)
pk = power(Omegam)
dpk = jax.jacfwd(power)(Omegam)
slope = np.asarray(Omegam * dpk / pk)  # d ln P / d ln Omega_m

## The check

A central finite difference of two more solves.

In [5]:
step = 1.0e-3
finite = np.asarray(
    Omegam * jnp.log(power(Omegam + step) / power(Omegam - step)) / (2.0 * step)
)

The two slopes agree to better than $10^{-2}$.

In [6]:
print(
    f"{'k [1/Mpc]':>12}  {'P(k) [Mpc^3]':>14}  {'dlnP/dlnOm':>11}  {'finite diff.':>12}"
)
for i in range(0, len(K), 7):
    print(f"{K[i]:12.4e}  {pk[i]:14.6e}  {slope[i]:11.5f}  {finite[i]:12.5f}")
print(f"largest difference between the two slopes: {np.abs(slope - finite).max():.1e}")

   k [1/Mpc]    P(k) [Mpc^3]   dlnP/dlnOm  finite diff.
  1.0000e-04    2.017395e+03     -1.53707      -1.53733
  2.7826e-04    5.402921e+03     -1.53109      -1.53113
  7.7426e-04    1.421706e+04     -1.49817      -1.49803
  2.1544e-03    3.486617e+04     -1.37571      -1.37572
  5.9948e-03    6.883071e+04     -1.03791      -1.03814
  1.6681e-02    7.392519e+04     -0.16228      -0.16243
  4.6416e-02    3.258203e+04      0.74318       0.74372
  1.2915e-01    7.019411e+03      1.50542       1.50645
  3.5938e-01    9.229130e+02      2.13948       2.14038
  1.0000e+00    9.044972e+01      2.42777       2.42849
largest difference between the two slopes: 7.4e-03
